In [1]:
from pyspark.sql import SparkSession
from howde import HoWDe_labelling
import yaml
import polars as pl

with open("../config.yaml") as fp:
    config = yaml.safe_load(fp)

/mnt/common-ssd/pintergreg/wp7_data_processing/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [2]:
PARTITION = 0  # valid between 0 and 255

In [3]:
data = pl.read_parquet(f"{config['dirs']['output']}/stops/{PARTITION:02x}.parquet")
data = data.rename(
    {
        "device_id": "useruuid",
        "stop_id": "loc",
        "interval_start": "start",
        "interval_stop": "end",
    }
).with_columns(
    country=pl.lit("HU"),
    tz_hour_start=pl.col("start").dt.base_utc_offset().dt.total_hours()
    + pl.col("start").dt.dst_offset().dt.total_hours(),
    tz_minute_start=pl.col("start").dt.base_utc_offset().dt.total_minutes() % 60,
)

In [4]:
data.head(3)

useruuid,loc,start,end,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,country,tz_hour_start,tz_minute_start
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,str,i64,i64
1,1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,17,"""HU""",2,0
1,2,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,14,"""HU""",2,0
1,3,2024-10-05 17:49:22 CEST,2024-10-05 17:59:54 CEST,10m 32s,47.476742,19.075131,0.0,0.0,3,8,"""HU""",2,0


In [5]:
# Initialize Spark session
spark = SparkSession.builder.appName("HoWDeApp").getOrCreate()

# Load your stop location data
input_data = spark.createDataFrame(data.to_pandas())

# Run HoWDe labelling
labeled_data = HoWDe_labelling(
    input_data,
    range_window_home=28,
    range_window_work=42,
    C_hours=0.4,
    C_days_H=0.4,
    C_days_W=0.5,
    f_hours_H=0.7,
    f_hours_W=0.4,
    f_days_W=0.6,
    output_format="stop",
    verbose=False,
)

# Show the results
labeled_data.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 09:55:21 WARN Utils: Your hostname, ab-vsrv09, resolves to a loopback address: 127.0.1.1; using 10.50.152.25 instead (on interface ens160)
26/04/17 09:55:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 09:55:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/17 09:56:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/17 09:56:15 WARN TaskSetManager: Stage 0 contains a task of very large size (1533 KiB). The maximum recommended task size is 1000 KiB.

+--------+-------+---+-------------------+----------+----------+-------------+------------+------------+
|useruuid|country|loc|               date|     start|       end|location_type|detect_H_loc|detect_W_loc|
+--------+-------+---+-------------------+----------+----------+-------------+------------+------------+
|       1|     HU|  1|2024-10-02 00:00:00|1727846646|1727847329|            O|        NULL|        NULL|
|       1|     HU|  2|2024-10-05 00:00:00|1728119203|1728120177|            O|        NULL|        NULL|
|       1|     HU|  3|2024-10-05 00:00:00|1728143362|1728143994|            O|        NULL|        NULL|
|       1|     HU|  4|2024-10-05 00:00:00|1728165234|1728165599|            O|        NULL|        NULL|
|       1|     HU|  4|2024-10-06 00:00:00|1728165600|1728165867|            O|        NULL|        NULL|
|       1|     HU|  5|2024-10-09 00:00:00|1728442014|1728442716|            O|        NULL|        NULL|
|       1|     HU|  6|2024-10-15 00:00:00|1728988969|17

In [7]:
pdf = labeled_data.toPandas()

26/04/17 09:57:51 WARN TaskSetManager: Stage 98 contains a task of very large size (1533 KiB). The maximum recommended task size is 1000 KiB.


In [ ]:
len(pdf[pdf["detect_H_loc"].notna()]) / len(pdf)

0.037611450187625795

In [ ]:
len(pdf[pdf["detect_W_loc"].notna()]) / len(pdf)

0.001653532472439182